# Naive Bayes classifier

In [1]:
import sys
import numpy as np
import struct
import matplotlib.pyplot as plt
import math
from tqdm import tqdm, trange

## Variables

In [9]:
train_images_filepath = "train-images.idx3-ubyte__"
train_labels_filepath = "train-labels.idx1-ubyte__"
test_images_filepath = "t10k-images.idx3-ubyte__"
test_labels_filepath = "t10k-labels.idx1-ubyte__"
toggle = 1

## Load Data

In [3]:
def LoadImages(filepath):
    with open(filepath,'rb') as f:
        magic, size = struct.unpack(">II", f.read(8))
        nrows, ncols = struct.unpack(">II", f.read(8))
        data = np.fromfile(f, dtype=np.dtype(np.uint8).newbyteorder('>'))
        data = data.reshape((size, nrows, ncols))

    return data

def LoadLabels(filepath):
    with open(filepath,'rb') as f:
        magic, size = struct.unpack(">II", f.read(8))
        data = np.fromfile(f, dtype=np.dtype(np.uint8).newbyteorder('>'))
        # data = data.reshape((size,))

    return data

In [4]:
x_train = LoadImages(train_images_filepath)
x_test = LoadImages(test_images_filepath)
y_train = LoadLabels(train_labels_filepath)
y_test = LoadLabels(test_labels_filepath)

## Training

In [5]:
def OutputPosterior(posterior):
    n_class = len(posterior)
    highest_prob = 1
    prediction = 0

    print("Posterior (in log scale):")

    for label in range(n_class):
        prob = posterior[label]
        print(label, ": ", prob)
        if prob < highest_prob:
            highest_prob = prob
            prediction = label
    
    return prediction

In [6]:
def Discrete(x_train, y_train, x_test, y_test):
    n_train, n_test = len(x_train), len(x_test)
    n_row, n_col = x_train[0].shape
    classes = sorted(np.unique(y_test)) # label numbers (0~9)
    n_class = len(classes) # how many label numbers (10)
    n_bin = 32
    peudocount = 0.00000001
    
    label_count = np.zeros(n_class)
    likelihood = np.zeros((n_class, n_row*n_col, n_bin)) # likelihood[label][pixels][bin]

    for i in trange(n_train):
        label_count[y_train[i]] += 1 # how many training label is belonging to this label number
        for row in range(n_row):
            for col in range(n_col):
                bin = x_train[i][row][col] // 8
                likelihood[y_train[i]][row*n_col + col][bin] += 1

    # compute prior prob for each label number(theda)
    # which is the prob of each label number amount all label number
    prior = label_count / n_train

    for label in range(n_class):
        for pixel in range(n_row*n_col):
            for bin in range(n_bin):
                if likelihood[label][pixel][bin] == 0:
                    likelihood[label][pixel][bin] = peudocount # 如果這個 label number 的所有資料的這個 pixel 都沒人屬於這個 bin, 給他一個極小值
                else:
                    likelihood[label][pixel][bin] /= label_count[label] # 這個 label number 的所有資料的這個 pixel 是屬於這個 bin 的比例
    
    error = 0
    for i in range(n_test):
        posterior = np.zeros(n_class)
        for label in range(n_class):
            posterior[label] = math.log(prior[label])
            for row in range(n_row):
                for col in range(n_col):
                    bin = x_test[i][row][col] // 8
                    posterior[label] += math.log(likelihood[label][row*n_col + col][bin]) # don't need to compute prob of marginal prob
        
        posterior /= sum(posterior)
        prediction = OutputPosterior(posterior)
        print("Prediction: ", prediction, ", Ans: ", y_test[i])
        print()

        if prediction != y_test[i]:
            error += 1

    error /= n_test

    # Print out the imagination of numbers in your Bayes classifier
    print("Imagination of numbers in Bayesian classifier:")

    for label in range(n_class):
        print()
        print(label, ":")
        for row in range(n_row):
            for col in range(n_col):
                # max of bins of likelihood[label][pixel] happened in 0 ~ 15 -> 0, 16 ~ 31 -> 1
                if np.argmax(likelihood[label][row*n_col + col]) <= 15:
                    print("0 ", end="")
                else:
                    print("1 ", end="")
            
            print()

    print("Error rate: ", error)

    return

In [7]:
def Continuous(x_train, y_train, x_test, y_test):
    n_train, n_test = len(x_train), len(x_test)
    n_row, n_col = x_train[0].shape
    classes = sorted(np.unique(y_test))
    n_class = len(classes)
    peudocount = 0.5 * math.pi
    
    label_count = np.zeros(n_class)
    mean = np.zeros((n_class, n_row, n_col))
    varience = np.zeros((n_class, n_row, n_col))

    for i in trange(n_train):
        label = y_train[i]
        label_count[label] += 1
        for row in range(n_row):
            for col in range(n_col):
                mean[label][row][col] += x_train[i][row][col]

    prior = label_count / n_train # compute prior prob for each label(theda)

    for label in range(n_class):
        mean[label] /= label_count[label]
    
    for i in trange(n_train):
        label = y_train[i]
        for row in range(n_row):
            for col in range(n_col):
                varience[label][row][col] += (x_train[i][row][col] - mean[label][row][col])**2

    for label in range(n_class):
        varience[label] /= label_count[label]
    
    for label in range(n_class):
        for row in range(n_row):
            for col in range(n_col):
                if varience[label][row][col] == 0:
                    varience[label][row][col] = peudocount

    error = 0
    for i in range(n_test):
        posterior = np.zeros(n_class)
        for label in range(n_class):
            posterior[label] += math.log(prior[label])
            for row in range(n_row):
                for col in range(n_col):
                    posterior[label] += -0.5*(math.log(2) + math.log(math.pi) + math.log(varience[label][row][col]) + (x_test[i][row][col] - mean[label][row][col])**2 / varience[label][row][col])
        
        posterior /= sum(posterior)
        prediction = OutputPosterior(posterior)
        print("Prediction: ", prediction, ", Ans: ", y_test[i])
        print()

        if prediction != y_test[i]:
            error += 1

    error /= n_test

    # Print out the imagination of numbers in your Bayes classifier
    print("Imagination of numbers in Bayesian classifier:")

    for label in range(n_class):
        print()
        print(label, ":")
        for row in range(n_row):
            for col in range(n_col):
                # mean[label][row][col] in 0~127 -> 0, 128 ~ 255 -> 1
                if mean[label][row][col] <= 127:
                    print("0 ", end="")
                else:
                    print("1 ", end="")
            
            print()

    print("Error rate: ", error)

    return

In [10]:
# plt.imshow(x_test[0,:,:], cmap='gray')
# plt.show()
# print(y_test[0])

if toggle == 0:
    Discrete(x_train, y_train, x_test, y_test)
else:
    Continuous(x_train, y_train, x_test, y_test)

100%|██████████| 60000/60000 [00:35<00:00, 1676.34it/s]


Posterior (in log scale):
0 :  0.04949909447238023
1 :  0.17614009546960444
2 :  0.051536010615219105
3 :  0.005176024680083611
4 :  0.004624098717475695
5 :  0.004140702760684055
6 :  0.6967595280916203
7 :  0.0031609643173380266
8 :  0.005716848834882383
9 :  0.003246632040711943
Prediction:  7 , Ans:  7

Posterior (in log scale):
0 :  0.018788669647995426
1 :  0.00782536674740311
2 :  0.0015700062322250618
3 :  0.2992431225718037
4 :  0.010892395671109306
5 :  0.003994385699083616
6 :  0.004154658043194435
7 :  0.43382115074810623
8 :  0.01484980783355344
9 :  0.20486043680552554
Prediction:  2 , Ans:  2

Posterior (in log scale):
0 :  6.339894850866699e-05
1 :  4.295013373720109e-05
2 :  6.000346788564732e-05
3 :  5.812141956256456e-05
4 :  5.849900583833497e-05
5 :  5.8860877356590626e-05
6 :  5.718982862146217e-05
7 :  0.9994769570884019
8 :  5.684172259160023e-05
9 :  6.717750749605306e-05
Prediction:  1 , Ans:  1

Posterior (in log scale):
0 :  0.047023791705371173
1 :  0.43313

# Online Learning

## Variables

In [14]:
filepath = "testfile.txt"
a = 10
b = 1

## Load Data

In [12]:
with open(filepath, 'r') as file:
    data = file.read()

data = data.split('\n')
data.remove('')

## Training

In [15]:
def Factorial(n):
    F = 1
    for i in range(1, n+1):
        F *= i

    return F

def C(a, b):
    return Factorial(a) / (Factorial(b) * Factorial(a-b))

def Binomial(N, m):
    p = m / N
    return C(N, m) * p**m * (1-p)**(N-m)

for line, line_data in enumerate(data):
    N = len(line_data)
    m = line_data.count("1")

    print(f"case {line+1}: {line_data}")
    print(f"Likelihood: {Binomial(N, m)}")
    print(f"Beta prior:     a = {a}, b = {b}")
    a += m
    b += N-m
    print(f"Beta posterior: a = {a}, b = {b}")
    print()

case 1: 0101010101001011010101
Likelihood: 0.16818809509277344
Beta prior:     a = 10, b = 1
Beta posterior: a = 21, b = 12

case 2: 0110101
Likelihood: 0.29375515303997485
Beta prior:     a = 21, b = 12
Beta posterior: a = 25, b = 15

case 3: 010110101101
Likelihood: 0.2286054241794335
Beta prior:     a = 25, b = 15
Beta posterior: a = 32, b = 20

case 4: 0101101011101011010
Likelihood: 0.18286870706509092
Beta prior:     a = 32, b = 20
Beta posterior: a = 43, b = 28

case 5: 111101100011110
Likelihood: 0.2143070548857833
Beta prior:     a = 43, b = 28
Beta posterior: a = 53, b = 33

case 6: 101110111000110
Likelihood: 0.20659760529408
Beta prior:     a = 53, b = 33
Beta posterior: a = 62, b = 39

case 7: 1010010111
Likelihood: 0.25082265600000003
Beta prior:     a = 62, b = 39
Beta posterior: a = 68, b = 43

case 8: 11101110110
Likelihood: 0.2619678932864457
Beta prior:     a = 68, b = 43
Beta posterior: a = 76, b = 46

case 9: 01000111101
Likelihood: 0.23609128871506807
Beta prior: 